<a href="https://colab.research.google.com/github/Guerrero-America/Optimizaci-n-no-lineal/blob/main/metodo_de_newton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np

def f(x):
    """Función"""
    x1, x2 = x
    return 100 * (x2 - x1**2)**2 + (1 - x1)**2

def gradiente(x):
    """Gradiente de la función"""
    x1, x2 = x
    df_dx1 = -400 * x1 * (x2 - x1**2) - 2 * (1 - x1)
    df_dx2 = 200 * (x2 - x1**2)
    return np.array([df_dx1, df_dx2])

def hessiana(x):
    """Hessiana de la función"""
    x1, x2 = x
    d2f_dx1_2 = -400 * (x2 - x1**2) + 800 * x1**2 + 2
    d2f_dx2_2 = 200.0
    d2f_dx1_dx2 = -400 * x1
    return np.array([[d2f_dx1_2, d2f_dx1_dx2],
                     [d2f_dx1_dx2, d2f_dx2_2]])

def newton_rosenbrock(x0, num_iteraciones=2):
    """
    Método de Newton para minimizar la función

    Parámetros:
    x0 : punto inicial (tupla o lista de 2 elementos)
    num_iteraciones : número de iteraciones a realizar

    Retorna:
    historial : lista con los puntos y valores en cada iteración
    """
    x = np.array(x0, dtype=float)
    historial = []

    print("=" * 60)
    print("  MÉTODO DE NEWTON ")
    print("=" * 60)
    print(f"f(x1, x2) = 100(x2 - x1²)² + (1 - x1)²")
    print(f"Punto inicial: ({x[0]}, {x[1]})")
    print(f"Número de iteraciones: {num_iteraciones}\n")

    for k in range(num_iteraciones):
        print("-" * 60)
        print(f"ITERACIÓN {k + 1}")
        print("-" * 60)

        print(f"Punto actual: x = ({x[0]:.10f}, {x[1]:.10f})")
        print(f"f(x) = {f(x):.10f}")

        g = gradiente(x)
        print(f"Gradiente: ∇f = ({g[0]:.10f}, {g[1]:.10f})")

        H = hessiana(x)
        print(f"Hessiana: H = [[{H[0,0]:.10f}, {H[0,1]:.10f}]")
        print(f"           [{H[1,0]:.10f}, {H[1,1]:.10f}]]")

        # Resolver H * Δx = -g
        try:
            delta = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            print("Error: Hessiana singular, no se puede resolver")
            break

        print(f"Dirección de Newton: Δx = ({delta[0]:.10f}, {delta[1]:.10f})")

        # Actualizar punto
        x = x + delta

        print(f"Nuevo punto: x = ({x[0]:.10f}, {x[1]:.10f})")
        print(f"Nuevo valor: f(x) = {f(x):.10f}")

        historial.append({
            'iteracion': k + 1,
            'x': x.copy(),
            'f': f(x),
            'gradiente': g,
            'delta': delta
        })

        print()

    print("=" * 60)
    print(f"RESULTADO FINAL después de {num_iteraciones} iteraciones:")
    print(f"x* = ({x[0]:.10f}, {x[1]:.10f})")
    print(f"f(x*) = {f(x):.10f}")
    print("=" * 60)

    return x, historial

def newton_con_verificacion(x0, tolerancia=1e-8, max_iter=100):
    """
    Versión con criterio de convergencia (itera hasta que el gradiente sea pequeño)
    """
    x = np.array(x0, dtype=float)

    print("=" * 60)
    print("  MÉTODO DE NEWTON CON CRITERIO DE CONVERGENCIA")
    print("=" * 60)
    print(f"Punto inicial: ({x[0]}, {x[1]})")
    print(f"Tolerancia: {tolerancia}\n")
    print("Iter\t\t x1\t\t x2\t\t f(x)\t\t ||∇f||")
    print("-" * 80)

    for i in range(max_iter):
        g = gradiente(x)
        norma_grad = np.linalg.norm(g)

        print(f"{i+1}\t\t{x[0]:.8f}\t{x[1]:.8f}\t{f(x):.8f}\t{norma_grad:.2e}")

        if norma_grad < tolerancia:
            print(f"\n Convergencia alcanzada en {i+1} iteraciones.")
            break

        H = hessiana(x)

        try:
            delta = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            print("Error: Hessiana singular")
            break

        x = x + delta

    print("-" * 80)
    print(f"\nRESULTADO FINAL:")
    print(f"x* = ({x[0]:.10f}, {x[1]:.10f})")
    print(f"f(x*) = {f(x):.10f}")

    return x

def newton_con_backtracking(x0, tolerancia=1e-8, max_iter=100, alpha_inicial=1.0, rho=0.5, c=1e-4):
    """
    Método de Newton con búsqueda lineal (backtracking) para mayor robustez
    """
    x = np.array(x0, dtype=float)

    print("=" * 60)
    print("  MÉTODO DE NEWTON CON BÚSQUEDA LINEAL")
    print("=" * 60)
    print(f"Punto inicial: ({x[0]}, {x[1]})")
    print(f"Tolerancia: {tolerancia}\n")
    print("Iter\t\t x1\t\t x2\t\t f(x)\t\t alpha")
    print("-" * 80)

    for i in range(max_iter):
        g = gradiente(x)
        norma_grad = np.linalg.norm(g)
        fx = f(x)

        print(f"{i+1}\t\t{x[0]:.8f}\t{x[1]:.8f}\t{fx:.8f}\t\t", end="")

        if norma_grad < tolerancia:
            print(f"\n Convergencia alcanzada en {i+1} iteraciones.")
            break

        H = hessiana(x)

        try:
            direccion = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            print("Error: Hessiana singular, usando dirección de gradiente")
            direccion = -g

        # Búsqueda lineal con backtracking (Armijo)
        alpha = alpha_inicial
        while f(x + alpha * direccion) > fx + c * alpha * np.dot(g, direccion):
            alpha *= rho
            if alpha < 1e-10:
                break

        print(f"{alpha:.6f}")

        x = x + alpha * direccion

    print("-" * 80)
    print(f"\nRESULTADO FINAL:")
    print(f"x* = ({x[0]:.10f}, {x[1]:.10f})")
    print(f"f(x*) = {f(x):.10f}")

    return x

# EJECUCIÓN PRINCIPAL
if __name__ == "__main__":
    # Punto inicial según el problema
    x0 = (-1.2, 1.0)

    # Opción 1: Exactamente 2 iteraciones (como pide el problema)
    print("\n" * 2)
    x_final, historial = newton_rosenbrock(x0, num_iteraciones=2)

    print("\n" * 2)
    print("RESUMEN DE LAS ITERACIONES:")
    print("-" * 50)
    for h in historial:
        print(f"Iteración {h['iteracion']}: x = ({h['x'][0]:.6f}, {h['x'][1]:.6f}), f = {h['f']:.6f}")

    # Opción 2: Con criterio de convergencia (opcional)
    # print("\n" * 3)
    # newton_con_verificacion(x0, tolerancia=1e-8)

    # Opción 3: Con búsqueda lineal (más robusto, opcional)
    # print("\n" * 3)
    # newton_con_backtracking(x0, tolerancia=1e-8)




  MÉTODO DE NEWTON 
f(x1, x2) = 100(x2 - x1²)² + (1 - x1)²
Punto inicial: (-1.2, 1.0)
Número de iteraciones: 2

------------------------------------------------------------
ITERACIÓN 1
------------------------------------------------------------
Punto actual: x = (-1.2000000000, 1.0000000000)
f(x) = 24.2000000000
Gradiente: ∇f = (-215.6000000000, -88.0000000000)
Hessiana: H = [[1330.0000000000, 480.0000000000]
           [480.0000000000, 200.0000000000]]
Dirección de Newton: Δx = (0.0247191011, 0.3806741573)
Nuevo punto: x = (-1.1752808989, 1.3806741573)
Nuevo valor: f(x) = 4.7318843253

------------------------------------------------------------
ITERACIÓN 2
------------------------------------------------------------
Punto actual: x = (-1.1752808989, 1.3806741573)
f(x) = 4.7318843253
Gradiente: ∇f = (-4.6378164146, -0.1222067921)
Hessiana: H = [[1107.2725665951, 470.1123595506]
           [470.1123595506, 200.0000000000]]
Dirección de Newton: Δx = (1.9383957701, -4.5557080121)
Nue

In [10]:
import matplotlib.pyplot as plt

# Graficar la trayectoria del método
def graficar_trayectoria(historial):
    x_vals = [h['x'][0] for h in historial]
    y_vals = [h['x'][1] for h in historial]

    plt.figure(figsize=(8, 6))
    plt.plot(x_vals, y_vals, 'ro-', linewidth=2, markersize=8)
    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title('Trayectoria del Método de Newton')
    plt.grid(True)
    plt.show()